# pandas 之六：时间序列处理

---

有一类数据叫**时间序列**：每天的销售额、每分钟的访问量、每年的产量……
它们共同的特点是**每一条都挂着一个时间点**。

pandas 是**处理时间序列最强的工具**，没有之一。

## 学习目标

1. 认识 3 个时间类型：`Timestamp` / `Timedelta` / `Period`
2. 会解析时间（字符串、整数、多列组合）
3. 会用 `.dt` 访问器**拆解**时间（提取年/月/日/周几/季度）
4. 会用 `date_range` **生成**时间序列
5. 掌握 `resample` **重采样**（本篇重点）
6. 会做时间切片和时区处理

> ⚠️ **重要版本提醒**：本篇的 `resample` 频率别名会用到
> **pandas 3.0 的新写法**。如果你看到别人教程里写 `'M'`、`'Q'`、`'Y'`，
> 在这台环境上会**直接报错**，原因见下文。


In [ ]:
import pandas as pd
import numpy as np

print('pandas 版本：', pd.__version__)


---
## 一、三个时间类型，先分清

| 类型 | 表示什么 | 例子 |
|---|---|---|
| `Timestamp` | **一个时间点** | `2023-08-15 10:00:00` |
| `Timedelta` | **一段时间差** | `5 days` |
| `Period` | **一个时间区间** | `2023-08`（代表整个 8 月） |

**类比**：`Timestamp` 是"几点"，`Timedelta` 是"多久"，`Period` 是"哪一段"。


In [ ]:
d = pd.Timestamp('2023-08-15 10:00')
print('d =', d)
print('type =', type(d))

# Timedelta
td = pd.Timedelta(days=3, hours=5)
print('\ntd =', td)
print('type =', type(td))

# Period
p = pd.Period('2023-08', freq='M')
print('\np =', p)
print('type =', type(p))
print('这个 Period 覆盖的区间：', p.start_time, '->', p.end_time)


---
## 二、解析时间：把各种格式变成 Timestamp

### 2.1 从字符串解析


In [ ]:
print(pd.to_datetime('2023-08-15'))
print(pd.to_datetime('2023/08/15 10:30'))
print(pd.to_datetime('2023-08-15 10:30:45'))


### ⚠️ 中文日期"认不出来"，必须给 format

很多人以为 `'2023年8月15日'` 这种写法 pandas 应该自动识别 ——
**实际上不行**，会抛 `DateParseError`。


In [ ]:
try:
    pd.to_datetime('2023年8月15日')
except Exception as e:
    print('直接解析中文日期 -> 报错：')
    print(str(e)[:100])

# 正确做法：用 format 把中文"字"也当占位符写进去
print('\n给 format 之后就能解析：')
print(pd.to_datetime('2023年8月15日', format='%Y年%m月%d日'))


> **经验**：凡是**格式不标准**的日期字符串（含中文、含特殊分隔符），
> **不要指望 pandas 猜**，直接给 `format` 最稳、最快。
> 自动推断是"尽力而为"，遇到异形格式就会失败。


### 2.2 从整数解析 —— **必须给 format**

像 `20030603` 这种"连在一起"的日期，pandas 猜不出来，要显式告诉它格式。

`%Y` = 年，`%m` = 月，`%d` = 日。


In [ ]:
df = pd.DataFrame({
    'sale': [100, 200, 300],
    'date': [20030603, 20030604, 20030605],
})
print('解析前：')
print(df)
print('date 列的类型：', df['date'].dtype)

# 用 format 指定格式
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')
print('\n解析后：')
print(df)
print('date 列的类型：', df['date'].dtype, '  <- 变成了 datetime64[ns]')


> **`datetime64[ns]` 是什么？**
> 这是 pandas 专门的时间类型（底层是"从 1970-01-01 起的纳秒数"）。
> **只有转成这个类型，才能用 `.dt` 和 `resample` 这些时间功能。**
>
> 如果一列还是 `object`（字符串），你做 `resample` 会直接报错 ——
> 这是新手最常见的错误之一。


### 2.3 解析失败怎么办？`errors` 参数

| 取值 | 行为 |
|---|---|
| `'raise'` | 报错（默认） |
| `'coerce'` | **转不了的变成 `NaT`**（推荐做数据清洗） |
| `'ignore'` | 原样留着 |


In [ ]:
bad_dates = pd.Series(['2023-01-01', '不是日期', '2023-03-01'])
print("errors='coerce' 会把无法解析的变成 NaT：")
print(pd.to_datetime(bad_dates, errors='coerce'))


### 2.4 从"多列拼成"一个日期

数据里常见"年、月、日、时"分在 4 列的情况。


In [ ]:
combo = pd.DataFrame({
    'year': [2023, 2024, 2025],
    'month': [1, 2, 3],
    'day': [1, 2, 3],
    'hour': [0, 12, 23],
    'minute': [0, 30, 59],
    'second': [0, 0, 0],
})
print('原始：')
print(combo)

combo['datetime'] = pd.to_datetime(combo)
print('\n拼成一列时间：')
print(combo[['datetime']])


**`pd.to_datetime(DataFrame)` 会自动去找名叫 year/month/day 的列来拼。**
前提是**列名必须是 pandas 认识的**（`year`/`month`/`day`/`hour`/`minute`/`second`）。


---
## 三、`.dt` 访问器：把时间"拆开"

**只要一列是 datetime 类型，就能用 `.dt` 访问器**，
它提供了一堆属性来提取时间的不同部分。


In [ ]:
dates = pd.Series(pd.to_datetime(['2023-01-15', '2023-06-30', '2023-12-31']))
dt = dates.dt

print('原始：')
print(dates)

print('\n=== 提取各部分 ===')
print('年   :', dt.year.tolist())
print('季   :', dt.quarter.tolist())
print('月   :', dt.month.tolist())
print('日   :', dt.day.tolist())
print('周几 :', dt.dayofweek.tolist(), '  <- 0=周一，6=周日')
print('周几名:', dt.day_name().tolist())

print('\n=== 判断类属性 ===')
print('是否月末:', dt.is_month_end.tolist())
print('是否月初:', dt.is_month_start.tolist())
print('是否年末:', dt.is_year_end.tolist())


### `.dt` 的三类东西

| 类型 | 例子 | 说明 |
|---|---|---|
| **属性** | `.year` `.month` `.day` `.quarter` `.dayofweek` | 不加括号 |
| **方法** | `.day_name()` `.month_name()` `.to_period()` | 要加括号 |
| **判断** | `.is_month_end` `.is_leap_year` | 返回布尔 |


In [ ]:
print('常用方法的完整清单演示：')
for name in ['day_name', 'month_name']:
    print(f'  {name}()  -> {getattr(dt, name)().tolist()}')

print('  to_period("M") ->', dt.to_period('M').tolist(), '  <- 变成"2023-01"这样的月份区间')
print('  to_period("Q") ->', dt.to_period('Q').tolist())
print('  to_period("Y") ->', dt.to_period('Y').tolist())


---
## 四、时间差：`Timedelta`

**两个 Timestamp 相减，得到一个 Timedelta。**


In [ ]:
d1 = pd.Timestamp('2023-01-01')
d2 = pd.Timestamp('2023-01-05')

delta = d2 - d1
print('两个时间相减：', delta)
print('type =', type(delta))
print('天数 =', delta.days)
print('总秒数 =', delta.total_seconds())


### 常用时间单位换算


In [ ]:
print('pd.Timedelta 可以构造各种时间差：')
for spec in ['1D', '3D', '2h', '30min', '45s', '1W']:
    print(f'  Timedelta("{spec}") = {pd.Timedelta(spec)}')


### 时间加减：直接 `+` / `-`

**`Timestamp ± Timedelta = Timestamp`**，可以整列一起算（这是 pandas 的威力）。


In [ ]:
df = pd.DataFrame({'date': pd.date_range('2023-01-01', periods=5, freq='D')})
print('原始：')
print(df)

df['plus_7d'] = df['date'] + pd.Timedelta(days=7)
df['minus_7d'] = df['date'] - pd.Timedelta(days=7)
print('\n加减 7 天：')
print(df)


### 更"聪明"的偏移：`DateOffset` / `offsets`

`Timedelta` 只懂"固定的时长"（7 天就是 7×24 小时）。
但有些需求是**日历概念**的，比如"下一个月末"—— 这就不固定了。

这时候要用 `pd.DateOffset` / `pd.offsets` 系列。


In [ ]:
df2 = pd.DataFrame({'date': pd.to_datetime(['2023-01-15', '2023-02-15', '2023-03-15'])})

df2['下月末'] = df2['date'] + pd.offsets.MonthEnd(n=1)
df2['下季末'] = df2['date'] + pd.offsets.QuarterEnd(n=1)
print(df2)


> **`Timedelta` vs `DateOffset`**：
> - `Timedelta(days=7)` -> 就是 7 天后（绝对时长）
> - `MonthEnd(n=1)` -> **当月最后一天**（跟着日历走）
>
> 做财务、报表时，"月末""季末"这类需求**必须用 offsets**。


---
## 五、`date_range`：生成时间序列

这个函数在"补全缺失日期""创建时间索引"时必不可少。


In [ ]:
# 1) 指定起止 + 频率
print('1) 按天，1月1日到1月10日：')
print(pd.date_range('2023-01-01', '2023-01-10', freq='D').tolist())

# 2) 指定起点 + 个数
print('\n2) 从1月1日起，连续 5 天：')
print(pd.date_range('2023-01-01', periods=5, freq='D').tolist())

# 3) 指定起点 + 个数 + 步长
print('\n3) 每 3 天一个，共 5 个：')
print(pd.date_range('2023-01-01', periods=5, freq='3D').tolist())


### 常用频率代码（**注意 pandas 3.0 的变更**）

| 想要 | 新写法（pandas 3.0） | 老写法（**已失效**） |
|---|---|---|
| 每天 | `'D'` | 同 |
| 每小时 | `'h'` | `'H'` |
| **月末** | **`'ME'`** | ~~`'M'`~~ ❌ |
| **季末** | **`'QE'`** | ~~`'Q'`~~ ❌ |
| **年末** | **`'YE'`** | ~~`'Y'`~~ ❌ |
| 每周一 | `'W-MON'` | 同 |

**规律：凡是"月末/季末/年末"这种"E 结尾"的（End），新版本都要求显式写上 `E`。**

为什么要改？因为老写法 `'M'` 有歧义 —— 它到底是"月"还是"月末"？
新版本干脆拆成 `'ME'`（月末）和 `'MS'`（月初），
逼你把意图写清楚，避免出错。


In [ ]:
print('月末 ME :', pd.date_range('2023-01-01', '2023-06-30', freq='ME').tolist())
print('月初 MS :', pd.date_range('2023-01-01', '2023-06-30', freq='MS').tolist())
print('季末 QE :', pd.date_range('2023-01-01', '2023-12-31', freq='QE').tolist())
print('年末 YE :', pd.date_range('2023-01-01', '2025-12-31', freq='YE').tolist())
print('\n每周一 W-MON :')
print(pd.date_range('2023-01-01', '2023-02-01', freq='W-MON').tolist())


### ⚠️ 亲眼看看老写法会怎么报错


In [ ]:
for old in ['M', 'Q', 'Y']:
    try:
        pd.date_range('2023-01-01', periods=3, freq=old)
        print(f"freq='{old}' -> 还能用")
    except ValueError as e:
        print(f"freq='{old}' -> ❌ 报错：{str(e)[:90]}")


---
## 六、`resample`：时间序列的核心操作

**`resample` = 按时间"重新分组"，然后做聚合。**

类比：`resample` 之于时间，就像 `groupby` 之于普通列。

```
日数据 --resample('ME').sum()--> 月数据
月数据 --resample('YE').mean()--> 年数据
```

**它是"降采样"（把密的变稀）的主要手段。**

先造一份 500 天的销售数据：


In [ ]:
np.random.seed(42)

# 生成 500 天，到"今天"为止
end_date = pd.Timestamp.today().normalize()
date_series = pd.date_range(end=end_date, periods=500, freq='D')

# 生成销售额：正态分布 + 截断 + 保留 2 位小数（模拟真实金额）
amounts = np.random.normal(loc=2000, scale=800, size=500)
amounts = np.clip(amounts, 100, 5000)
amounts = np.round(amounts, 2)

sales = pd.DataFrame({'日期': date_series, '销售额': amounts}).set_index('日期')
print('数据范围：', sales.index.min().date(), '->', sales.index.max().date())
print('总行数：', len(sales))
print('\n前 5 行：')
print(sales.head())


In [ ]:
print("=== 按月汇总（'ME' 月末）===")
monthly = sales.resample('ME').sum()
print(monthly.tail(6))

print("\n=== 按季汇总（'QE' 季末）===")
print(sales.resample('QE').sum().tail(4))


In [ ]:
print("=== 按年汇总（'YE' 年末）===")
print(sales.resample('YE').sum())

print("\n=== 按周汇总（'W-MON'，以周一为每周起点）===")
print(sales.resample('W-MON').sum().tail(4))

print("\n=== 每 10 天汇总 ===")
print(sales.resample('10D').sum().head(4))


### `to_period()`：让索引更好看

`resample('ME')` 之后的索引是"月末那一天"（比如 `2024-01-31`）。
如果只关心"哪个月"，可以转成 Period：


In [ ]:
monthly_raw = sales.resample('ME').sum()
print('转之前（索引是具体日期）：')
print(monthly_raw.index[-3:])

# ⚠️ 注意：resample 用 'ME'，但 to_period 要用 'M'（下面的坑专门讲）
monthly_period = monthly_raw.to_period('M')
print('\n转之后（索引变成"年月"）：')
print(monthly_period.index[-3:])
print('\n数值不变，只是索引表达方式变了：')
print(monthly_period.tail(3))


### ⚠️ 一个极其隐蔽的坑：`resample` 要 `'ME'`，`to_period` 要 `'M'`

这是 pandas 3.0 里**最容易把人绕晕**的地方：

| 用法 | 正确写法 | 错误写法 |
|---|---|---|
| `resample(...)` | `'ME'` | ~~`'M'`~~ ❌ |
| `.to_period(...)` | **`'M'`** | ~~`'ME'`~~ ❌ |

**为什么？** 因为 `Period` 表示的是"一个区间"（整个 8 月），
它本身就是"月"这个周期，**不需要再区分月初还是月末**。
而 `resample` 是"在时间轴上切点"，就必须说清切在月末还是月初。

所以同一个环境里，**两个函数用两套别名**，写错了会直接报错。


In [ ]:
try:
    monthly_raw.to_period('ME')
except ValueError as e:
    print('to_period 用 "ME" -> 报错：')
    print(str(e)[:110])

print('\n用 "M" 才是对的：')
print(monthly_raw.to_period('M').index[-3:])


### `resample` 不只 `.sum()`，还能做各种聚合

任何能用在 `groupby` 上的聚合，`resample` 都能用：


In [ ]:
q = sales.resample('QE')
print('季末 求和    :', q.sum()['销售额'].tail(3).tolist())
print('季末 平均    :', q.mean()['销售额'].tail(3).round(2).tolist())
print('季末 最大值  :', q.max()['销售额'].tail(3).tolist())
print('季末 记录数  :', q.count()['销售额'].tail(3).tolist())


In [ ]:
# agg 一次算多个
print('一次算「总和 + 平均 + 最大」：')
print(sales.resample('YE').agg({'销售额': ['sum', 'mean', 'max']}).round(1))


### `ohlc()`：金融场景专用

O-H-L-C = Open(开) / High(高) / Low(低) / Close(收)。
就是把每个时间段的**第一个、最大、最小、最后一个**值都取出来。


In [ ]:
print('按月算 OHLC：')
print(sales.resample('ME').ohlc().tail(3))


**理解一下**：以一个月为例，
- `open` = 这个月第一天的销售额
- `high` = 这个月最高的一天
- `low` = 这个月最低的一天
- `close` = 这个月最后一天

这是股票 K 线图的数据结构。


---
## 七、时间切片与筛选

把时间列**设为索引**后，pandas 支持用**字符串直接切片**，非常方便。


In [ ]:
start = sales.index.min().date()
mid = (sales.index.min() + pd.Timedelta(days=200)).date()
end = sales.index.max().date()
print('数据从', start, '到', end, '，取前 201 天：')

subset = sales[str(start):str(mid)]
print('子集形状：', subset.shape)
print(subset.head(3))


### 几种常见的时间筛选


In [ ]:
df_t = pd.DataFrame({
    'date': pd.date_range('2023-01-01', periods=100, freq='D'),
    'value': range(100),
})
df_t['date'] = pd.to_datetime(df_t['date'])

# 方法 1：布尔条件
print('1) 条件筛选：')
print(df_t[(df_t['date'] >= '2023-01-15') & (df_t['date'] <= '2023-01-20')][['date', 'value']].head(3))

# 方法 2：between
print('\n2) between：')
print(df_t[df_t['date'].between('2023-01-15', '2023-01-20')][['date', 'value']].head(3))

# 方法 3：按年/月/季度筛选
print('\n3) 按月份筛（1 月）：')
print(df_t[df_t['date'].dt.month == 1]['value'].count(), '条记录')

print('4) 按周几筛（周末）：')
print(df_t[df_t['date'].dt.dayofweek >= 5]['value'].count(), '条记录')


### 设为索引后可以直接用字符串切片（最简洁）


In [ ]:
df_idx = df_t.set_index('date')
print('用字符串切片直接取一段：')
print(df_idx['2023-01-15':'2023-01-20'].head(3))


---
## 八、时区处理（进阶）

**⚠️ 最容易踩的坑：不能拿"带时区的时间"和"不带时区的时间"比较**，会直接报错。


In [ ]:
tz = pd.DataFrame({'date': pd.to_datetime(['2023-08-15 10:00', '2023-08-16 10:00'])})
print('原始（无时区，叫 "naive"）：')
print(tz)
print('dtype =', tz['date'].dtype)

# 加上时区（叫 "aware"）
tz['utc'] = tz['date'].dt.tz_localize('UTC')
tz['shanghai'] = tz['utc'].dt.tz_convert('Asia/Shanghai')
print('\n加时区 + 转换时区：')
print(tz)


注意：UTC 的 `10:00` 转成上海时间变成了 `18:00`（+8 小时），
**但这是同一个时刻**，只是"看的方式"不同。


In [ ]:
print('dtype 变化：')
print('  naive    :', tz['date'].dtype)
print('  aware(utc):', tz['utc'].dtype)


### ⚠️ 演示：naive 和 aware 混用会报错


In [ ]:
try:
    # 右边是没有时区的 Timestamp，左边是带时区的，不能比
    bad = tz[tz['utc'] > pd.Timestamp.now()]
except TypeError as e:
    print('报错了（正确行为）：')
    print(str(e)[:150])


**正确做法：两边都带时区，或者两边都不带。**


In [ ]:
# 给 now() 指定时区，两边就都是 aware 了
now_utc = pd.Timestamp.now(tz='UTC')
print('用于比较的当前时间（带 UTC 时区）：', now_utc)
recent = tz[tz['utc'] < now_utc]
print('\n筛选成功，得到', len(recent), '行')


> **经验法则**：
> - 数据入库/存储时统一用 **UTC**（避免跨时区混乱）
> - 展示给用户时再 `tz_convert` 到本地时区
> - **不要混用 naive 和 aware**，会报错


---
## 九、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 时间列还是 `object` 就 `resample` | 报错 | 先 `pd.to_datetime()` |
| 整数日期不写 `format` | 解析出乱七八糟的日期 | `format='%Y%m%d'` |
| `freq='M'` / `'Q'` / `'Y'` | pandas 3.0 **直接报错** | 改用 `'ME'` / `'QE'` / `'YE'` |
| `to_period('ME')` | 报 "please use 'M' instead of 'ME'" | **`to_period` 仍用 `'M'` / `'Q'` / `'Y'`** |
| `freq='H'`（大写） | 已废弃 | 用小写 `'h'` |
| naive 和 aware 时间比较 | `TypeError` | 两边都带时区或都不带 |
| `dt.year` 写成 `dt.year()` | 报错 | 属性**不加括号** |
| `dt.day_name` 忘了括号 | 拿到的是方法对象 | 方法**要加括号**：`dt.day_name()` |
| `Timedelta` 当 `DateOffset` 用 | "月末"算成"30 天后" | 月末/季末用 `pd.offsets.*` |
| `resample` 后忘了聚合 | 拿到一个 Resampler 对象 | 后面要接 `.sum()` / `.mean()` 等 |
| 解析失败没设 `errors` | 整批报错中断 | 数据清洗时用 `errors='coerce'` |

---

## 十、动手练习

**练习 1**：把 `'2023-08-15 10:30:00'` 转成 Timestamp，打印它的类型、年份、月份、周几。

**练习 2**：整数 `20230815` 怎么转成日期？写出代码。

**练习 3**：生成 2023 年全年的月末日期列表（提示：`freq='ME'`）。

**练习 4**：计算 `2023-01-01` 到 `2023-12-31` 之间相差多少天、多少小时。

**练习 5**：有一份 2023 年每天的销售数据，按季度求总销售额和平均销售额。

**练习 6**：给一个日期 `2023-02-10`，算出它的"本月最后一天"和"本季度最后一天"。

**练习 7**：从一份时间序列里，筛选出所有"周六和周日"的数据。

---

**答案思路（先自己做）**

```python
# 1
d = pd.Timestamp('2023-08-15 10:30:00')
print(type(d), d.year, d.month, d.day_name())

# 2
print(pd.to_datetime('20230815', format='%Y%m%d'))

# 3
print(pd.date_range('2023-01-01', '2023-12-31', freq='ME'))

# 4
delta = pd.Timestamp('2023-12-31') - pd.Timestamp('2023-01-01')
print(delta.days, delta.total_seconds() / 3600)

# 5
# sales.resample('QE').agg({'销售额': ['sum', 'mean']})

# 6
d = pd.Timestamp('2023-02-10')
print(d + pd.offsets.MonthEnd(0))       # n=0 表示"本"月末
print(d + pd.offsets.QuarterEnd(0))

# 7
# df[df.index.dayofweek >= 5]
```
